In [0]:
%sql
-- Register the project's S3 path using the existing storage credential.
CREATE EXTERNAL LOCATION places_ratings_s3
URL 's3://alumno-cpomares-datos/Places_Ratings/'
WITH (STORAGE CREDENTIAL `cred-cpomares`)
COMMENT 'S3 storage for the Places Ratings bootcamp project';

In [0]:
%sql
-- Inspect the registered S3 external location.
DESCRIBE EXTERNAL LOCATION places_ratings_s3;

In [0]:
%sql
-- Create the project's catalog and landing schema.
CREATE CATALOG IF NOT EXISTS places_ratings
COMMENT 'Business exploration project for La Paz and El Alto';

CREATE SCHEMA IF NOT EXISTS places_ratings.landing
COMMENT 'File landing area for source ingestion';

In [0]:
%sql
-- Create an external volume for the project's source files in S3.
CREATE EXTERNAL VOLUME IF NOT EXISTS places_ratings.landing.source_files
LOCATION 's3://alumno-cpomares-datos/Places_Ratings/landing/'
COMMENT 'Source files and extraction metadata for the Places Ratings project';

In [0]:
# Verify writing and reading a small test file through the external volume.

import json
from uuid import uuid4
from datetime import datetime, timezone

test_path = (
    "/Volumes/places_ratings/landing/source_files/"
    f"storage_check_{uuid4().hex}.json"
)

test_data = {
    "project": "places_ratings",
    "purpose": "storage_connectivity_test",
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

with open(test_path, "w", encoding="utf-8") as file:
    json.dump(test_data, file, indent=2)

with open(test_path, "r", encoding="utf-8") as file:
    recovered_data = json.load(file)

if recovered_data != test_data:
    raise ValueError("Storage check failed: recovered content differs.")

print("Write and read check passed.")
print(f"Test file: {test_path}")

In [0]:
%sql
-- Create the Bronze schema for traceable source ingestion.
CREATE SCHEMA IF NOT EXISTS places_ratings.bronze
COMMENT 'Original API responses and source ingestion metadata';